In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
processed_path = '/content/drive/MyDrive/FraudDetection_project/processed_data/'
models_path = '/content/drive/MyDrive/FraudDetection_project/saved_models/'
os.makedirs(models_path, exist_ok=True)

print("Loading datasets and model artifacts...")

# Sparkov Enriched (from Day 6)
sparkov_train = pd.read_csv(processed_path + 'sparkov_train_enriched.csv')
sparkov_test = pd.read_csv(processed_path + 'sparkov_test_enriched.csv')

# PaySim Processed (from Day 4 - already SMOTEd and encoded)
paysim_train = pd.read_csv(processed_path + 'paysim_train_processed.csv')
paysim_test = pd.read_csv(processed_path + 'paysim_test_processed.csv')

# Load the Day 4 Encoder directly!
sparkov_encoder = joblib.load(models_path + 'label_encoders.joblib')

print("Data loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading datasets and model artifacts...
Data loaded successfully!


In [3]:
print("Preparing Sparkov data for classification...")

# 1. Apply Day 4 Encoders to text columns
cat_cols = ['category', 'gender', 'job']
sparkov_train[cat_cols] = sparkov_encoder.transform(sparkov_train[cat_cols])
sparkov_test[cat_cols] = sparkov_encoder.transform(sparkov_test[cat_cols])

# 2. Drop non-predictive columns (identities, raw coordinates, dates)
sparkov_drop = ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'first', 'last',
                'street', 'city', 'state', 'zip', 'dob', 'trans_num', 'lat', 'long', 'merch_lat', 'merch_long']
sparkov_drop = [c for c in sparkov_drop if c in sparkov_train.columns]

X_train_sp_raw = sparkov_train.drop(columns=sparkov_drop + ['is_fraud'])
y_train_sp_raw = sparkov_train['is_fraud']

X_test_sp = sparkov_test.drop(columns=sparkov_drop + ['is_fraud'])
y_test_sp = sparkov_test['is_fraud']

# 3. Apply SMOTE to the enriched training data (10% ratio)
print("Balancing Sparkov classes with SMOTE... (this may take a minute)")
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_sp, y_train_sp = smote.fit_resample(X_train_sp_raw, y_train_sp_raw)

# 4. Train the XGBoost Classifier
print("Training Sparkov XGBoost Classifier... (this will take a few minutes)")
# We use scale_pos_weight to give extra attention to the fraud class
xgb_sparkov = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=10,
    random_state=42,
    n_jobs=-1
)
xgb_sparkov.fit(X_train_sp, y_train_sp)

print("Sparkov Model Training Complete!")

Preparing Sparkov data for classification...
Balancing Sparkov classes with SMOTE... (this may take a minute)
Training Sparkov XGBoost Classifier... (this will take a few minutes)
Sparkov Model Training Complete!


In [4]:
print("Preparing PaySim data for classification...")

# 1. Separate Features and Target
X_train_ps = paysim_train.drop(columns=['isFraud'])
y_train_ps = paysim_train['isFraud']

X_test_ps = paysim_test.drop(columns=['isFraud'])
y_test_ps = paysim_test['isFraud']

# 2. Train the XGBoost Classifier
print("Training PaySim XGBoost Classifier... (this will take a few minutes)")
xgb_paysim = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=10,
    random_state=42,
    n_jobs=-1
)
xgb_paysim.fit(X_train_ps, y_train_ps)

print("PaySim Model Training Complete!")

Preparing PaySim data for classification...
Training PaySim XGBoost Classifier... (this will take a few minutes)
PaySim Model Training Complete!


In [5]:
print("Saving Trained Classifiers to Drive...")

joblib.dump(xgb_sparkov, models_path + 'xgb_sparkov_model.joblib')
joblib.dump(xgb_paysim, models_path + 'xgb_paysim_model.joblib')

# We will also save the test sets for tomorrow's Evaluation phase
X_test_sp['is_fraud'] = y_test_sp
X_test_sp.to_csv(processed_path + 'sparkov_ready_for_eval.csv', index=False)

X_test_ps['isFraud'] = y_test_ps
X_test_ps.to_csv(processed_path + 'paysim_ready_for_eval.csv', index=False)

print("Day 7 Deliverables successfully saved!")
print("The Brain of your Fraud Detection System is officially online.")

Saving Trained Classifiers to Drive...
Day 7 Deliverables successfully saved!
The Brain of your Fraud Detection System is officially online.
